In [1]:
import ee

# Inicializa la sesión de Earth Engine
try:
    ee.Initialize()
    print("Google Earth Engine inicializado correctamente.")
except Exception as e:
    print("No se pudo inicializar. Ejecutando autenticación...")
    ee.Authenticate()
    ee.Initialize()
    print("Autenticación e inicialización completadas.")

/home/wmlegion/miniconda3/envs/agri_land_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Google Earth Engine inicializado correctamente.


In [6]:
def get_sar_moisture_by_year_and_quarter(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    report = {}

    # Colección Sentinel-1 GRD
    s1 = ee.ImageCollection("COPERNICUS/S1_GRD") \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filterBounds(region)

    for year in range(start_year, end_year + 1):
        report[str(year)] = {}
        for q in range(1, 5):
            q_name = f'Q{q}'
            start_month = (q - 1) * 3 + 1
            end_month = q * 3
            
            start_date = f'{year}-{start_month:02d}-01'
            # Calcular fecha fin (manejando diciembre)
            next_month = end_month + 1
            next_year = year
            if next_month > 12:
                next_month = 1
                next_year += 1
            end_date = f'{next_year}-{next_month:02d}-01'
            
            # Filtrar periodo y aplicar filtro de ruido (speckle)
            period_col = s1.filterDate(start_date, end_date)
            
            if period_col.size().getInfo() > 0:
                # Aplicar filtro de mediana para reducir ruido speckle
                image = period_col.select('VV').median()
                
                # Reducción espacial en la hectárea
                stats = image.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=region,
                    scale=10,
                    bestEffort=True
                ).get('VV')
                
                report[str(year)][q_name] = stats.getInfo() if stats.getInfo() is not None else 0
            else:
                report[str(year)][q_name] = 0
                
    return report

# Ejemplo de ejecución:
# start = 2024
# end = 2026
# sar_data = get_sar_moisture_by_year_and_quarter(7.3297, -73.1867, start, end)
# print(sar_data)

In [10]:
start = 2020
end = 2026
sar_data = get_sar_moisture_by_year_and_quarter(7.3297, -73.1867, start, end)

import json
print(json.dumps(sar_data, indent=4))

{
    "2020": {
        "Q1": -7.814380693914505,
        "Q2": -7.693119379728277,
        "Q3": -7.519251767276359,
        "Q4": -7.576142798030222
    },
    "2021": {
        "Q1": -7.9003775242481655,
        "Q2": -7.702234947235258,
        "Q3": -7.372373964113974,
        "Q4": -8.185770871844495
    },
    "2022": {
        "Q1": -9.123813193845534,
        "Q2": -8.653944914692282,
        "Q3": -9.034840727449156,
        "Q4": -9.059483765011755
    },
    "2023": {
        "Q1": -9.316814832296824,
        "Q2": -9.46804116250733,
        "Q3": -9.19055427420895,
        "Q4": -8.653285390960033
    },
    "2024": {
        "Q1": -9.137371390501537,
        "Q2": -9.014815806400328,
        "Q3": -9.157568898113853,
        "Q4": -8.793960411681903
    },
    "2025": {
        "Q1": -8.788835965681617,
        "Q2": -8.54468503283798,
        "Q3": -8.433902092081551,
        "Q4": -8.216525363436316
    },
    "2026": {
        "Q1": -8.237899180919621,
        "Q2": -7

In [11]:
import ee

def get_lst_by_year_and_quarter(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    report = {}

    # Colección Landsat 8 Collection 2 Level 2 (Surface Temperature)
    lst_col = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterBounds(region)

    for year in range(start_year, end_year + 1):
        report[str(year)] = {}
        for q in range(1, 5):
            q_name = f'Q{q}'
            start_month = (q - 1) * 3 + 1
            end_month = q * 3
            start_date = f'{year}-{start_month:02d}-01'
            next_month = end_month + 1
            next_year = year
            if next_month > 12: next_month = 1; next_year += 1
            end_date = f'{next_year}-{next_month:02d}-01'
            
            period_col = lst_col.filterDate(start_date, end_date)
            
            if period_col.size().getInfo() > 0:
                # Banda ST_B10 es la temperatura de superficie
                # Escalar: (valor * 0.00341802 + 149) - 273.15 para Celsius
                def apply_scale(image):
                    temp = image.select('ST_B10').multiply(0.00341802).add(149).subtract(273.15)
                    return temp.rename('LST')
                
                stats = period_col.map(apply_scale).median().reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=region,
                    scale=30,
                    bestEffort=True
                ).get('LST')
                
                report[str(year)][q_name] = stats.getInfo() if stats.getInfo() is not None else 0
            else:
                report[str(year)][q_name] = 0
    return report

In [12]:
start = 2020
end = 2026
lst_data = get_lst_by_year_and_quarter(7.3297, -73.1867, start, end)

import json
print(json.dumps(lst_data, indent=4))

{
    "2020": {
        "Q1": 29.933820781589404,
        "Q2": -53.998045294377086,
        "Q3": 22.281321781304488,
        "Q4": -2.7371787431508245
    },
    "2021": {
        "Q1": 25.686822566988845,
        "Q2": -2.552426169363106,
        "Q3": 21.114968761882704,
        "Q4": 23.730244805536334
    },
    "2022": {
        "Q1": 11.916438211329632,
        "Q2": 20.00495578720952,
        "Q3": 17.116678762614544,
        "Q4": 7.990254590125717
    },
    "2023": {
        "Q1": 25.415117664860357,
        "Q2": 29.31583560493298,
        "Q3": -25.32532192496926,
        "Q4": 14.044359501349179
    },
    "2024": {
        "Q1": 23.841040210262587,
        "Q2": -123.14852013999999,
        "Q3": 20.25329164840784,
        "Q4": 16.209153398184384
    },
    "2025": {
        "Q1": 18.66962273321789,
        "Q2": 0.08246768600839657,
        "Q3": -12.01033860971507,
        "Q4": 2.3677864101424766
    },
    "2026": {
        "Q1": 27.559836422720696,
        "Q2": 2

In [13]:
import ee

def get_lst_by_year_and_quarter(lat, lon, start_year, end_year):
    """
    Extrae la Temperatura de Superficie (LST) de Landsat 8/9.
    Incluye filtro de rango térmico [-10, 60] grados Celsius para eliminar artefactos.
    """
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    report = {}

    # Colección Landsat 8 Collection 2 Level 2 (Surface Temperature)
    lst_col = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterBounds(region)

    for year in range(start_year, end_year + 1):
        report[str(year)] = {}
        for q in range(1, 5):
            q_name = f'Q{q}'
            # Calcular rangos de meses para el trimestre
            start_month = (q - 1) * 3 + 1
            end_month = q * 3
            start_date = f'{year}-{start_month:02d}-01'
            
            next_month = end_month + 1
            next_year = year
            if next_month > 12: 
                next_month = 1
                next_year += 1
            end_date = f'{next_year}-{next_month:02d}-01'
            
            period_col = lst_col.filterDate(start_date, end_date)
            
            # Función interna con filtro de rango (Actualización implementada)
            def apply_scale_and_filter(image):
                # Conversión de Kelvin escalado a Celsius
                temp = image.select('ST_B10').multiply(0.00341802).add(149).subtract(273.15)
                # Aplicar máscara: solo conservar valores entre -10 y 60 grados C
                return temp.updateMask(temp.gt(-10).And(temp.lt(60))).rename('LST')
            
            if period_col.size().getInfo() > 0:
                # Procesamiento: aplicar filtro, tomar mediana temporal y reducir a hectárea
                stats = period_col.map(apply_scale_and_filter).median().reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=region,
                    scale=30,
                    bestEffort=True
                ).get('LST')
                
                # Validar resultado
                val = stats.getInfo()
                report[str(year)][q_name] = val if val is not None else 0
            else:
                report[str(year)][q_name] = 0
                
    return report

# Ejemplo de uso:
# lst_data = get_lst_by_year_and_quarter(lat_value, lon_value, 2020, 2026)
# print(lst_data)

In [14]:
start = 2020
end = 2026
lst_data = get_lst_by_year_and_quarter(7.3297, -73.1867, start, end)

import json
print(json.dumps(lst_data, indent=4))

{
    "2020": {
        "Q1": 29.933820781589404,
        "Q2": 28.3223002545475,
        "Q3": 24.830884480949738,
        "Q4": 13.18187036943577
    },
    "2021": {
        "Q1": 25.686822566988845,
        "Q2": 12.191908186606167,
        "Q3": 24.772400058290533,
        "Q4": 23.730244805536334
    },
    "2022": {
        "Q1": 17.3135705585503,
        "Q2": 20.00495578720952,
        "Q3": 24.503199476402255,
        "Q4": 13.027813142854772
    },
    "2023": {
        "Q1": 25.415117664860357,
        "Q2": 29.594550312324042,
        "Q3": 12.005630393000015,
        "Q4": 28.846834568469284
    },
    "2024": {
        "Q1": 23.841040210262587,
        "Q2": 26.27313520552516,
        "Q3": 21.93952885146929,
        "Q4": 16.209153398184384
    },
    "2025": {
        "Q1": 21.174636124983255,
        "Q2": 30.953187483960907,
        "Q3": 11.838598056977666,
        "Q4": 17.212491209838007
    },
    "2026": {
        "Q1": 27.785879250960914,
        "Q2": 23.65097

In [17]:
import ee

def get_lst_by_year_and_month(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    report = {}

    lst_col = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterBounds(region)

    for year in range(start_year, end_year + 1):
        report[str(year)] = {}
        # Bucle de meses del 1 al 12
        for month in range(1, 13):
            month_name = f'M{month:02d}'
            
            # Definir inicio y fin de mes
            start_date = f'{year}-{month:02d}-01'
            
            # Cálculo de fin de mes
            next_month = month + 1
            next_year = year
            if next_month > 12:
                next_month = 1
                next_year += 1
            end_date = f'{next_year}-{next_month:02d}-01'
            
            period_col = lst_col.filterDate(start_date, end_date)
            
            # Función de filtrado y conversión
            def apply_scale_and_filter(image):
                temp = image.select('ST_B10').multiply(0.00341802).add(149).subtract(273.15)
                return temp.updateMask(temp.gt(-10).And(temp.lt(60))).rename('LST')
            
            if period_col.size().getInfo() > 0:
                stats = period_col.map(apply_scale_and_filter).median().reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=region,
                    scale=30,
                    bestEffort=True
                ).get('LST')
                
                val = stats.getInfo()
                report[str(year)][month_name] = val if val is not None else 0
            else:
                report[str(year)][month_name] = 0
                
    return report

In [18]:
start = 2020
end = 2026
lst_data = get_lst_by_year_and_month(7.3297, -73.1867, start, end)

import json
print(json.dumps(lst_data, indent=4))

{
    "2020": {
        "M01": 28.155512335597788,
        "M02": 30.773034915860354,
        "M03": 29.557477207128514,
        "M04": 28.3223002545475,
        "M05": 0,
        "M06": 0,
        "M07": 17.36193124515645,
        "M08": 26.721239537033536,
        "M09": 24.151013569832415,
        "M10": -2.7371787431508245,
        "M11": 0,
        "M12": 29.10091948202237
    },
    "2021": {
        "M01": 25.68278873053633,
        "M02": 28.1335415321788,
        "M03": 21.05683568876539,
        "M04": 14.49357039296371,
        "M05": 23.27366989485477,
        "M06": -6.214998817083774,
        "M07": 22.338242793977685,
        "M08": 0,
        "M09": 24.916151668145268,
        "M10": 25.891121345930202,
        "M11": 13.601963167765385,
        "M12": 18.16283729968717
    },
    "2022": {
        "M01": 22.22485427687153,
        "M02": 20.98841752613689,
        "M03": 12.040995442949741,
        "M04": 18.66903651414528,
        "M05": 20.08591273661454,
        "M0

In [21]:
import ee

def get_lst_monthly_strict(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    report = {}

    # Landsat 8 Collection 2 Level 2 (Surface Temperature)
    # Usamos T1_L2 que tiene bandas de calidad (QA_PIXEL)
    lst_col = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterBounds(region)

    for year in range(start_year, end_year + 1):
        report[str(year)] = {}
        for month in range(1, 13):
            month_name = f'M{month:02d}'
            start_date = f'{year}-{month:02d}-01'
            next_month = month + 1
            next_year = year
            if next_month > 12: next_month = 1; next_year += 1
            end_date = f'{next_year}-{next_month:02d}-01'
            
            # Filtrar por fecha
            period_col = lst_col.filterDate(start_date, end_date)
            
            # MÁSCARA ESTRICTA: 
            # 1. Definimos los bits de nubes y sombras de nubes (QA_PIXEL)
            # 2. Solo promediamos los píxeles que son "limpios" (Clear)
            def clean_and_calc(image):
                qa = image.select('QA_PIXEL')
                # Bit 3: Cloud, Bit 4: Cloud Shadow
                mask = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
                
                temp = image.select('ST_B10').multiply(0.00341802).add(149).subtract(273.15)
                return temp.updateMask(mask).rename('LST')
            
            if period_col.size().getInfo() > 0:
                # Calculamos la media de los píxeles limpios
                mean_temp = period_col.map(clean_and_calc).mean().reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=region,
                    scale=30,
                    bestEffort=True
                ).get('LST')
                
                val = mean_temp.getInfo()
                # Solo guardamos si el valor es razonable (> 20C)
                report[str(year)][month_name] = val if (val is not None and val > 15) else None
            else:
                report[str(year)][month_name] = None # 'None' indica sin datos válidos
                
    return report

In [22]:
start = 2020
end = 2026
lst_data = get_lst_monthly_strict(7.3297, -73.1867, start, end)

import json
print(json.dumps(lst_data, indent=4))

{
    "2020": {
        "M01": 28.155512335597788,
        "M02": 30.773034915860354,
        "M03": 29.557477207128514,
        "M04": 28.3223002545475,
        "M05": null,
        "M06": null,
        "M07": null,
        "M08": 26.899672592838005,
        "M09": 24.151013569832415,
        "M10": null,
        "M11": null,
        "M12": 29.10091948202237
    },
    "2021": {
        "M01": 25.68958028956853,
        "M02": 28.1335415321788,
        "M03": 28.433608362273766,
        "M04": 27.87699430756986,
        "M05": 23.27366989485477,
        "M06": null,
        "M07": null,
        "M08": null,
        "M09": 28.717334574407836,
        "M10": 25.891121345930202,
        "M11": 26.363961926368734,
        "M12": 28.66739233748606
    },
    "2022": {
        "M01": 22.22485427687153,
        "M02": 30.18495407256426,
        "M03": null,
        "M04": 18.66903651414528,
        "M05": 20.08591273661454,
        "M06": 25.671689713078223,
        "M07": null,
        "M08

In [ ]:
import ee

def get_et_monthly(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    report = {}

    # Colección MODIS Evapotranspiración (8 días) [MODIS/006/MOD16A2]
    et_col = ee.ImageCollection("MODIS/061/MOD16A2") \
        .filterBounds(region) \
        .select('ET')

    for year in range(start_year, end_year + 1):
        report[str(year)] = {}
        for month in range(1, 13):
            month_name = f'M{month:02d}'
            start_date = f'{year}-{month:02d}-01'
            next_month = month + 1
            next_year = year
            if next_month > 12: next_month = 1; next_year += 1
            end_date = f'{next_year}-{next_month:02d}-01'
            
            # Filtramos el mes
            period_col = et_col.filterDate(start_date, end_date)
            
            if period_col.size().getInfo() > 0:
                # Aplicamos el factor de escala (0.1) y sumamos el mes
                # Esto da el total de ET mensual en mm o kg/m2
                et_monthly = period_col.sum().multiply(0.1)
                
                stats = et_monthly.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=region,
                    scale=500, # La resolución nativa de MODIS es 500m
                    bestEffort=True
                ).get('ET')
                
                val = stats.getInfo()
                report[str(year)][month_name] = val if val is not None else 0
            else:
                report[str(year)][month_name] = 0
    return report

In [6]:
start = 2021
end = 2026
et_data = get_et_monthly(7.3297, -73.1867, start, end)

import json
print(json.dumps(et_data, indent=4))

{
    "2021": {
        "M01": 128.1,
        "M02": 115.10000000000001,
        "M03": 112.10000000000001,
        "M04": 87,
        "M05": 104,
        "M06": 106.5,
        "M07": 123,
        "M08": 120.2,
        "M09": 122.10000000000001,
        "M10": 81.2,
        "M11": 117.5,
        "M12": 100.7
    },
    "2022": {
        "M01": 126.2,
        "M02": 119.4,
        "M03": 117.30000000000001,
        "M04": 80.5,
        "M05": 120.9,
        "M06": 112.10000000000001,
        "M07": 129,
        "M08": 118.7,
        "M09": 104.5,
        "M10": 50.5,
        "M11": 100.5,
        "M12": 107
    },
    "2023": {
        "M01": 117.9,
        "M02": 131.9,
        "M03": 119.30000000000001,
        "M04": 72.3,
        "M05": 114.80000000000001,
        "M06": 128,
        "M07": 131.20000000000002,
        "M08": 123.30000000000001,
        "M09": 123.9,
        "M10": 76.60000000000001,
        "M11": 108.4,
        "M12": 99.80000000000001
    },
    "2024": {
        